# Predicting joint contact forces from gait inputs: OpenSim JCF data, NIH ACL cohort

Runtime > Change runtime type > Python 3. CPU is enough for this notebook; a GPU comes later, for the model fitting.

## Setup

Everything used here comes preinstalled on Colab: `numpy` for arrays, `pandas` to read the CSV files, `matplotlib` to plot. The colours are the Okabe-Ito palette, readable for colour-blind readers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SPEED_COLOURS = {1.0: "#0072B2", 1.25: "#D55E00", 1.4: "#000000", 1.5: "#009E73"}  # one colour per belt speed
JOINT_COLOURS = {"hip": "#CC79A7", "knee": "#0072B2", "ankle": "#009E73"}

## Data

Four CSV files in the private GitHub repository `marysalvana/OpenSim-JCF-GP`, folder `data/`, with a `log.txt` that defines every column. Read `log.txt` once before anything else.

One row is one stride of the right leg. `x-*.csv` holds the inputs (three ground reaction force curves, five joint angle curves, belt speed, mass, height), `y-*.csv` the outputs (the hip, knee and ankle joint contact force curves from OpenSim). Each curve is sampled at 101 points of the gait cycle, 0 to 100 %, heel strike to heel strike. Row i of x and row i of y are the same stride.

The repository is private, so cloning needs a GitHub personal access token (GitHub > Settings > Developer settings > Fine-grained tokens, read access to this repository). Paste it when asked; it is not stored.

In [ ]:
import os
from getpass import getpass

if not os.path.isdir("OpenSim-JCF-GP"):
    token = getpass("GitHub token (leave empty if the repository is public): ")
    url = f"https://{token}@github.com/marysalvana/OpenSim-JCF-GP.git" if token else "https://github.com/marysalvana/OpenSim-JCF-GP.git"
    os.system(f"git clone -q {url}")
DATA = "OpenSim-JCF-GP/data"
sorted(os.listdir(DATA))

In [ ]:
x_train = pd.read_csv(f"{DATA}/x-train.csv")
y_train = pd.read_csv(f"{DATA}/y-train.csv")
x_test = pd.read_csv(f"{DATA}/x-test.csv")
y_test = pd.read_csv(f"{DATA}/y-test.csv")

In [ ]:
print(open(f"{DATA}/log.txt").read()[:3000])  # the first part of the data log; read the whole file in the repository

## Explore

A data frame with one row per stride. The first six columns identify the stride and give the scalar inputs; the rest are the curves, one column per gait point.

In [ ]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape  # rows = strides, columns = identifiers + curve points

In [ ]:
x_train.columns[:12].tolist()  # identifiers, scalar inputs, then the first gait points of the vertical GRF

In [ ]:
x_train.iloc[:5, :10]  # the first five strides

In [ ]:
y_train.columns[:8].tolist()  # same identifiers, then the hip JCF at gait points 0, 1, ...

In [ ]:
x_train["subject"].value_counts().sort_index()  # strides per training subject

In [ ]:
x_test["subject"].value_counts().sort_index()  # the three held-out subjects; never seen in training

In [ ]:
x_train["belt_speed_mps"].value_counts().sort_index()  # strides per belt speed; 1.4 m/s is one subject's fast trial

In [ ]:
x_train.groupby("subject")[["mass_kg", "height_m"]].first()  # one mass and height per subject

In [ ]:
(x_train[["subject", "block", "stride_in_block"]].values == y_train[["subject", "block", "stride_in_block"]].values).all()  # rows aligned

## From wide columns to curves

A curve called `name` occupies the 101 columns `name_t000` to `name_t100`. The helper returns a `(strides, 101)` array for one curve, and the gait grid is simply 0, 1, ..., 100 percent.

In [ ]:
GRID = np.arange(101)  # percent of the gait cycle

INPUT_CURVES = ["grf_vertical_BW", "grf_anteroposterior_BW", "grf_mediolateral_BW",
                "hip_flexion_deg", "hip_adduction_deg", "hip_rotation_deg", "knee_flexion_deg", "ankle_angle_deg"]
OUTPUT_CURVES = ["hip_jcf_BW", "knee_jcf_BW", "ankle_jcf_BW"]

def curve(df, name):
    return df[[f"{name}_t{t:03d}" for t in GRID]].to_numpy()  # (strides, 101)

In [ ]:
grf_v = curve(x_train, "grf_vertical_BW")
grf_v.shape

In [ ]:
grf_v[0, [0, 15, 50, 62, 100]]  # one stride: heel strike, loading peak, push-off peak, about toe-off, next heel strike

In [ ]:
knee = curve(y_train, "knee_jcf_BW")
knee.max(axis=1)[:10]  # peak knee JCF of the first ten strides, body weights

## Visualise the inputs

All 480 training strides of each input curve, coloured by belt speed, with the mean curve of each speed on top. Stance is roughly the first 62 % of the cycle, swing the rest. Look for: the two humps of the vertical GRF, braking then propulsion in the fore-aft GRF, hip extension until about 50 %, the knee flexion peak in swing, and the ankle push-off near 60 %.

In [ ]:
LABELS = {"grf_vertical_BW": "Vertical GRF [BW]", "grf_anteroposterior_BW": "Fore-aft GRF [BW]", "grf_mediolateral_BW": "Medio-lateral GRF [BW]",
          "hip_flexion_deg": "Hip flexion [deg]", "hip_adduction_deg": "Hip adduction [deg]", "hip_rotation_deg": "Hip rotation [deg]",
          "knee_flexion_deg": "Knee flexion [deg]", "ankle_angle_deg": "Ankle dorsiflexion [deg]"}

def overlay(ax, df, name, label):
    C = curve(df, name)
    speeds = df["belt_speed_mps"].to_numpy()
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].T, color=SPEED_COLOURS[v], lw=0.5, alpha=0.25)
    for v in sorted(SPEED_COLOURS):
        rows = speeds == v
        if rows.any():
            ax.plot(GRID, C[rows].mean(axis=0), color=SPEED_COLOURS[v], lw=2.5, label=f"{v:g} m/s (n={rows.sum()})")
    ax.axvline(62, color="grey", lw=0.8, ls=":")  # approximate toe-off
    ax.set_title(label, loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat, INPUT_CURVES):
    overlay(ax, x_train, name, LABELS[name])
axes[0][0].legend(frameon=False, fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("Gait cycle [%]")
fig.suptitle("Inputs: 480 training strides, 12 subjects, coloured by belt speed; dotted line = approximate toe-off", x=0.01, ha="left")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, name in zip(axes, ["grf_vertical_BW", "knee_flexion_deg", "ankle_angle_deg"]):
    overlay(ax, x_test, name, LABELS[name] + ", test subjects")
    ax.set_xlabel("Gait cycle [%]")
axes[0].legend(frameon=False, fontsize=9)
fig.tight_layout()

## Visualise the outputs

The joint contact force is the total force pressing the two bones of a joint together: the external load plus the compression produced by the muscles that cross the joint. That is why it reaches 3 to 6 body weights in walking while the ground reaction force is about 1 body weight. Two peaks in stance (loading response and push-off), near zero in swing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, name in zip(axes, OUTPUT_CURVES):
    overlay(ax, y_train, name, name.split("_")[0].capitalize() + " JCF [BW]")
    ax.set_xlabel("Gait cycle [%]")
axes[0].legend(frameon=False, fontsize=9)
fig.suptitle("Outputs: joint contact forces of the same 480 training strides", x=0.01, ha="left")
fig.tight_layout()

In [ ]:
peaks = pd.DataFrame({name.split("_")[0]: curve(y_train, name).max(axis=1) for name in OUTPUT_CURVES})
peaks.describe().round(2)  # peak JCF per stride, body weights; compare with published walking values (hip 2.5-5.5, knee 2-5.5, ankle 3.5-7)

## One stride, inputs and outputs together

The question the model has to answer: given the eight curves on the left and the scalars, produce the three curves on the right. Same stride, same gait axis.

In [ ]:
i = 0  # row index of the stride to show; change it
row = x_train.iloc[i]
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, name in zip(axes.flat[:8], INPUT_CURVES):
    ax.plot(GRID, curve(x_train, name)[i], color="#0b0b0b", lw=2)
    ax.set_title(LABELS[name], loc="left"); ax.set_xlim(0, 100); ax.grid(alpha=0.3)
fig.suptitle(f"Inputs of one stride: {row['subject']}, {row['block']}, stride {row['stride_in_block']}, {row['belt_speed_mps']:g} m/s, {row['mass_kg']} kg, {row['height_m']} m", x=0.01, ha="left")
fig.tight_layout()

fig, ax = plt.subplots(figsize=(8, 4.5))
for name in OUTPUT_CURVES:
    j = name.split("_")[0]
    ax.plot(GRID, curve(y_train, name)[i], color=JOINT_COLOURS[j], lw=2.5, label=j)
ax.set_title("Outputs of the same stride: joint contact forces", loc="left"); ax.set_xlabel("Gait cycle [%]"); ax.set_ylabel("JCF [BW]")
ax.set_xlim(0, 100); ax.grid(alpha=0.3); ax.legend(frameon=False)
fig.tight_layout()

## Subject to subject

The same joint, one panel per training subject, strides at 1.0 m/s only. Each subject has a recognisable shape of their own; that between-subject variation, not stride-to-stride noise, is what makes prediction for a new person hard.

In [ ]:
subjects = sorted(x_train["subject"].unique())
fig, axes = plt.subplots(3, 4, figsize=(20, 11), sharex=True, sharey=True)
for ax, s in zip(axes.flat, subjects):
    rows = (x_train["subject"] == s) & (x_train["belt_speed_mps"] == 1.0)
    ax.plot(GRID, knee[rows.to_numpy()].T, color=JOINT_COLOURS["knee"], lw=1, alpha=0.7)
    ax.plot(GRID, knee.mean(axis=0), color="grey", lw=1.5, ls="--")  # grand mean over all training strides and speeds
    ax.set_title(s.replace("NIH_ACL_", ""), loc="left"); ax.grid(alpha=0.3)
for ax in axes[2]:
    ax.set_xlabel("Gait cycle [%]")
for ax in axes[:, 0]:
    ax.set_ylabel("Knee JCF [BW]")
fig.suptitle("Knee JCF at 1.0 m/s, one panel per training subject (dashed grey: mean of all training strides)", x=0.01, ha="left")
fig.tight_layout()